In [3]:
import pandas as pd 
import json

df1 = pd.read_csv('mturk_results/Batch_5356502_batch_results.csv', encoding='utf-8')
df2 = pd.read_csv('mturk_results/Batch_5356519_batch_results.csv', encoding='utf-8')

# merge the two dataframes
df = pd.concat([df1, df2], ignore_index=True)
df.columns

Index(['HITId', 'HITTypeId', 'Title', 'Description', 'Keywords', 'Reward',
       'CreationTime', 'MaxAssignments', 'RequesterAnnotation',
       'AssignmentDurationInSeconds', 'AutoApprovalDelayInSeconds',
       'Expiration', 'NumberOfSimilarHITs', 'LifetimeInSeconds',
       'AssignmentId', 'WorkerId', 'AssignmentStatus', 'AcceptTime',
       'SubmitTime', 'AutoApprovalTime', 'ApprovalTime', 'RejectionTime',
       'RequesterFeedback', 'WorkTimeInSeconds', 'LifetimeApprovalRate',
       'Last30DaysApprovalRate', 'Last7DaysApprovalRate', 'Input.q_strings',
       'Answer.taskAnswers', 'Approve', 'Reject'],
      dtype='object')

In [4]:
submission_df = df[['WorkerId','WorkTimeInSeconds','Answer.taskAnswers']]
submission_df

,WorkerId,WorkTimeInSeconds,Answer.taskAnswers
0,A2H3SVUWH2L9ZO,981,"[{""104_B_ours_ar_q1"":{""ar"":false,""ours"":true},..."
1,A2N94XU62FQKB9,1207,"[{""074_B_ours_lam_q1"":{""lam"":false,""ours"":true..."
2,A2OXGAYFXOIVYS,679,"[{""074_A_ar_ours_q1"":{""ar"":false,""ours"":true},..."
3,A2UR7QQI8FBLE0,183,"[{""074_A_ours_hr_q1"":{""hr"":false,""ours"":true},..."
4,A1RWHQIX1934RI,1274,"[{""074_B_ours_4dgs_q1"":{""4dgs"":false,""ours"":tr..."
...,...,...,...
115,ADSEGF47JPK5L,374,"[{""074_A_lam_ours_q1"":{""lam"":false,""ours"":true..."
116,A19BQKCS0MF3A5,1391,"[{""104_B_gaga_ours_q1"":{""gaga"":false,""ours"":tr..."
117,A35J9IIFZC67I2,509,"[{""104_A_ar_ours_q1"":{""ar"":true,""ours"":false},..."
118,A1TK7V460ZKPYT,563,"[{""074_B_lam_ours_q1"":{""lam"":false,""ours"":true..."


In [6]:
# answer stats per worker
# recall: each worker answers 9 x 3 = 27 questions, full score is 27
for i in range(len(submission_df)):
    print(f"workerid: {submission_df.at[i, 'WorkerId']}, worktime: {submission_df.at[i, 'WorkTimeInSeconds']}")
    answer_json = json.loads(submission_df.at[i, 'Answer.taskAnswers'])
    methods = ['ours', 'ga', 'ar', '4dgs', 'lam', 'gaga', 'hr']
    score_dict = {method: 0 for method in methods}
    for question, answer in answer_json[0].items():
        # print(f"question: {question}, answer: {answer}") # see individual answers
        for method, score in answer.items():
            if method in methods:
                score_dict[method] += int(score)
    print(f"score_dict: {score_dict}")




workerid: A2H3SVUWH2L9ZO, worktime: 981
score_dict: {'ours': 16, 'ga': 2, 'ar': 1, '4dgs': 5, 'lam': 3, 'gaga': 0, 'hr': 0}
workerid: A2N94XU62FQKB9, worktime: 1207
score_dict: {'ours': 24, 'ga': 0, 'ar': 0, '4dgs': 3, 'lam': 0, 'gaga': 0, 'hr': 0}
workerid: A2OXGAYFXOIVYS, worktime: 679
score_dict: {'ours': 21, 'ga': 2, 'ar': 0, '4dgs': 0, 'lam': 0, 'gaga': 3, 'hr': 1}
workerid: A2UR7QQI8FBLE0, worktime: 183
score_dict: {'ours': 21, 'ga': 0, 'ar': 0, '4dgs': 3, 'lam': 0, 'gaga': 3, 'hr': 0}
workerid: A1RWHQIX1934RI, worktime: 1274
score_dict: {'ours': 17, 'ga': 3, 'ar': 1, '4dgs': 3, 'lam': 1, 'gaga': 0, 'hr': 2}
workerid: A3QTSAB6UOBSKM, worktime: 284
score_dict: {'ours': 21, 'ga': 3, 'ar': 0, '4dgs': 0, 'lam': 3, 'gaga': 0, 'hr': 0}
workerid: A2X9J2CBD5D5TU, worktime: 410
score_dict: {'ours': 21, 'ga': 0, 'ar': 5, '4dgs': 1, 'lam': 0, 'gaga': 0, 'hr': 0}
workerid: AUEJL4P5KPSXL, worktime: 1150
score_dict: {'ours': 16, 'ga': 3, 'ar': 1, '4dgs': 3, 'lam': 1, 'gaga': 3, 'hr': 0}
worker

In [7]:
# method score distribution by subj+A/B

# create df (workerid, worktime, subj, speech, methodA, methodB, methodA_wins, methodB_wins, ours_wins, ga_wins, ar_wins, 4dgs_wins, lam_wins, gaga_wins, hr_wins)
df_list = []
for i in range(len(submission_df)):
    workerId = submission_df.at[i, 'WorkerId']
    worktime = submission_df.at[i, 'WorkTimeInSeconds']

    answer_json = json.loads(submission_df.at[i, 'Answer.taskAnswers'])
    methods = ['ours', 'ga', 'ar', '4dgs', 'lam', 'gaga', 'hr']
    # score_dict = {method: 0 for method in methods}
    for question, answer in answer_json[0].items():
        subj = question.split('_')[0]
        speech = question.split('_')[1]
        methodA = question.split('_')[2]
        methodB = question.split('_')[3]
        question_id = question.split('_')[4]

        methodA_wins = answer[methodA]
        methodB_wins = answer[methodB]

        # using wins, get 0,1,0,0,0,0,0 for ours, ga, ar, 4dgs, lam, gaga, hr
        ours_wins = 0
        ga_wins = 0
        ar_wins = 0
        dgs4_wins = 0
        lam_wins = 0
        gaga_wins = 0
        hr_wins = 0
        
        if methodA == 'ours':
            ours_wins = int(methodA_wins)
        elif methodA == 'ga':
            ga_wins = int(methodA_wins)
        elif methodA == 'ar':
            ar_wins = int(methodA_wins)
        elif methodA == '4dgs':
            dgs4_wins = int(methodA_wins)
        elif methodA == 'lam':
            lam_wins = int(methodA_wins)
        elif methodA == 'gaga':
            gaga_wins = int(methodA_wins)
        elif methodA == 'hr':
            hr_wins = int(methodA_wins)
        
        if methodB == 'ours':
            ours_wins = int(methodB_wins)
        elif methodB == 'ga':
            ga_wins = int(methodB_wins)
        elif methodB == 'ar':
            ar_wins = int(methodB_wins)
        elif methodB == '4dgs':
            dgs4_wins = int(methodB_wins)
        elif methodB == 'lam':
            lam_wins = int(methodB_wins)
        elif methodB == 'gaga':
            gaga_wins = int(methodB_wins)
        elif methodB == 'hr':
            hr_wins = int(methodB_wins)

        # create tuple
        df_list.append((workerId, worktime, subj, speech, question_id, methodA, methodB, methodA_wins, methodB_wins, ours_wins, ga_wins, ar_wins, dgs4_wins, lam_wins, gaga_wins, hr_wins))

# create df
df = pd.DataFrame(df_list, columns=['WorkerId', 'WorkTimeInSeconds', 'subj', 'speech', 'q_id', 'methodA', 'methodB', 'methodA_wins', 'methodB_wins', 'ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins'])
df

,WorkerId,WorkTimeInSeconds,subj,speech,q_id,methodA,methodB,methodA_wins,methodB_wins,ours_wins,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins
0,A2H3SVUWH2L9ZO,981,104,B,q1,ours,ar,True,False,1,0,0,0,0,0,0
1,A2H3SVUWH2L9ZO,981,104,B,q2,ours,ar,False,True,0,0,1,0,0,0,0
2,A2H3SVUWH2L9ZO,981,104,B,q3,ours,ar,True,False,1,0,0,0,0,0,0
3,A2H3SVUWH2L9ZO,981,218,A,q1,ga,ours,True,False,0,1,0,0,0,0,0
4,A2H3SVUWH2L9ZO,981,218,A,q2,ga,ours,False,True,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,A26JZCXT4WWHNU,1148,306,A,q2,ours,4dgs,True,False,1,0,0,0,0,0,0
3236,A26JZCXT4WWHNU,1148,306,A,q3,ours,4dgs,True,False,1,0,0,0,0,0,0
3237,A26JZCXT4WWHNU,1148,460,B,q1,ours,ar,True,False,1,0,0,0,0,0,0
3238,A26JZCXT4WWHNU,1148,460,B,q2,ours,ar,True,False,1,0,0,0,0,0,0


In [8]:
df_binary = df.drop(columns=['methodA_wins', 'methodB_wins', 'methodA', 'methodB'])
df_binary

,WorkerId,WorkTimeInSeconds,subj,speech,q_id,ours_wins,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins
0,A2H3SVUWH2L9ZO,981,104,B,q1,1,0,0,0,0,0,0
1,A2H3SVUWH2L9ZO,981,104,B,q2,0,0,1,0,0,0,0
2,A2H3SVUWH2L9ZO,981,104,B,q3,1,0,0,0,0,0,0
3,A2H3SVUWH2L9ZO,981,218,A,q1,0,1,0,0,0,0,0
4,A2H3SVUWH2L9ZO,981,218,A,q2,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3235,A26JZCXT4WWHNU,1148,306,A,q2,1,0,0,0,0,0,0
3236,A26JZCXT4WWHNU,1148,306,A,q3,1,0,0,0,0,0,0
3237,A26JZCXT4WWHNU,1148,460,B,q1,1,0,0,0,0,0,0
3238,A26JZCXT4WWHNU,1148,460,B,q2,1,0,0,0,0,0,0


In [9]:
df_binary['WorkTimeInSeconds'].describe()

count    3240.000000
mean      872.008333
std       432.818449
min       134.000000
25%       505.500000
50%       819.000000
75%      1231.500000
max      1743.000000
Name: WorkTimeInSeconds, dtype: float64

In [10]:
df_binary.groupby(['WorkerId','WorkTimeInSeconds']).sum(['ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins']).reset_index()

,WorkerId,WorkTimeInSeconds,ours_wins,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins
0,A1391Q0GNLGNAH,323,27,0,0,0,0,0,0
1,A1391Q0GNLGNAH,525,27,0,0,0,0,0,0
2,A145C8GKNAG28E,441,27,0,0,0,0,0,0
3,A18WDR1RQHEQ7D,427,15,5,0,4,0,1,2
4,A19BQKCS0MF3A5,1391,27,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
115,APXK08LMHFHM8,824,25,0,1,0,0,1,0
116,AQ29XB1EZ954V,876,13,1,3,0,1,3,6
117,AUD9XPEM59ML1,358,17,0,3,1,0,3,3
118,AUEJL4P5KPSXL,1150,16,3,1,3,1,3,0


In [ ]:
# Q: what subject+AB did ours lose the most?
df_binary.drop(columns=['WorkTimeInSeconds']).groupby(['subj','speech']).sum(['ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins']).reset_index()

,subj,speech,ours_wins,ga_wins,talkg_wins,instag_wins
0,074,A,129,14,15,22
1,074,B,149,8,8,15
2,104,A,112,20,26,22
3,104,B,120,14,23,23
4,218,A,119,13,28,20
5,218,B,136,3,22,19
6,253,A,137,16,12,15
7,253,B,110,11,38,21
8,264,A,94,23,34,29
9,264,B,126,13,23,18


In [11]:
# Q: what question (1,2,3) did ours lose the most?
df_binary.drop(columns=['WorkTimeInSeconds']).groupby(['q_id']).sum(['ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins']).reset_index()

# finding: we win in realness, lip sync & quality debatable

,q_id,ours_wins,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins
0,q1,846,49,40,31,22,51,41
1,q2,839,46,44,34,30,46,41
2,q3,890,38,41,33,17,40,21


In [19]:
all_wins = df_binary.drop(columns=['WorkTimeInSeconds']).groupby(['q_id']).sum(['ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins']).reset_index()

all_wins = all_wins.drop(columns=['ours_wins'])
# divide all numbers in all_wins by 360
# 360 = 20 workers x 9 questions x 2 (A/B) for sigA
# 180 = 10 workers x 9 questions x 2 (A/B) for cvpr

num_participants_per_question = 10
num_subjects = 9
num_sentences = 2
num_comparisons_per_method = num_participants_per_question * num_subjects * num_sentences

all_wins['ours_wins_ga'] = 1 - all_wins['ga_wins'] / num_comparisons_per_method
all_wins['ours_wins_ar'] = 1 - all_wins['ar_wins'] / num_comparisons_per_method
all_wins['ours_wins_4dgs'] = 1 - all_wins['4dgs_wins'] / num_comparisons_per_method
all_wins['ours_wins_lam'] = 1 - all_wins['lam_wins'] / num_comparisons_per_method
all_wins['ours_wins_gaga'] = 1 - all_wins['gaga_wins'] / num_comparisons_per_method
all_wins['ours_wins_hr'] = 1 - all_wins['hr_wins'] / num_comparisons_per_method

all_wins

,q_id,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins,ours_wins_ga,ours_wins_ar,ours_wins_4dgs,ours_wins_lam,ours_wins_gaga,ours_wins_hr
0,q1,49,40,31,22,51,41,0.727778,0.777778,0.827778,0.877778,0.716667,0.772222
1,q2,46,44,34,30,46,41,0.744444,0.755556,0.811111,0.833333,0.744444,0.772222
2,q3,38,41,33,17,40,21,0.788889,0.772222,0.816667,0.905556,0.777778,0.883333


In [20]:
# flip the ours_wins to get ours_losses
all_wins['ours_losses_ga'] = 1 - all_wins['ours_wins_ga']
all_wins['ours_losses_ar'] = 1 - all_wins['ours_wins_ar']
all_wins['ours_losses_4dgs'] = 1 - all_wins['ours_wins_4dgs']
all_wins['ours_losses_lam'] = 1 - all_wins['ours_wins_lam']
all_wins['ours_losses_gaga'] = 1 - all_wins['ours_wins_gaga']
all_wins['ours_losses_hr'] = 1 - all_wins['ours_wins_hr']   
all_losses = all_wins.drop(columns=['ours_wins_ga', 'ours_wins_ar', 'ours_wins_4dgs', 'ours_wins_lam', 'ours_wins_gaga', 'ours_wins_hr'])
all_losses


,q_id,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins,ours_losses_ga,ours_losses_ar,ours_losses_4dgs,ours_losses_lam,ours_losses_gaga,ours_losses_hr
0,q1,49,40,31,22,51,41,0.272222,0.222222,0.172222,0.122222,0.283333,0.227778
1,q2,46,44,34,30,46,41,0.255556,0.244444,0.188889,0.166667,0.255556,0.227778
2,q3,38,41,33,17,40,21,0.211111,0.227778,0.183333,0.094444,0.222222,0.116667


In [22]:
# show ours_losses for each method in percentages
for method in ['ours_losses_ga', 'ours_losses_ar', 'ours_losses_4dgs', 'ours_losses_lam', 'ours_losses_gaga', 'ours_losses_hr']:
    all_losses[method] = all_losses[method] * 100
all_losses

,q_id,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins,ours_losses_ga,ours_losses_ar,ours_losses_4dgs,ours_losses_lam,ours_losses_gaga,ours_losses_hr
0,q1,49,40,31,22,51,41,27.222222,22.222222,17.222222,12.222222,28.333333,22.777778
1,q2,46,44,34,30,46,41,25.555556,24.444444,18.888889,16.666667,25.555556,22.777778
2,q3,38,41,33,17,40,21,21.111111,22.777778,18.333333,9.444444,22.222222,11.666667


In [21]:
# Q: what subject+AB+q did ours lose the most?
df_binary.drop(columns=['WorkTimeInSeconds']).groupby(['subj','speech','q_id']).sum(['ours_wins', 'ga_wins', 'ar_wins', '4dgs_wins', 'lam_wins', 'gaga_wins', 'hr_wins']).reset_index()

,subj,speech,q_id,ours_wins,ga_wins,ar_wins,4dgs_wins,lam_wins,gaga_wins,hr_wins
0,074,A,q1,49,1,1,2,1,3,3
1,074,A,q2,49,2,1,1,0,5,2
2,074,A,q3,53,2,2,2,0,0,1
3,074,B,q1,45,1,5,0,3,4,2
4,074,B,q2,45,2,4,2,1,5,1
5,074,B,q3,49,2,3,2,1,3,0
6,104,A,q1,45,1,4,2,0,4,4
7,104,A,q2,41,3,5,3,3,3,2
8,104,A,q3,53,0,3,3,0,0,1
9,104,B,q1,48,1,4,1,1,2,3


In [96]:
# sort by time
df_binary.sort_values(by=['WorkTimeInSeconds'], ascending=True, inplace=True)
df_binary

,WorkerId,WorkTimeInSeconds,subj,speech,q_id,ours_wins,ga_wins,talkg_wins,instag_wins
1145,A3VQDUUA2RI8KL,176,218,B,q3,0,1,0,0
1146,A3VQDUUA2RI8KL,176,218,B,q1,1,0,0,0
1139,A3VQDUUA2RI8KL,176,104,B,q3,0,0,0,1
1134,A3VQDUUA2RI8KL,176,074,A,q1,1,0,0,0
1140,A3VQDUUA2RI8KL,176,218,A,q1,1,0,0,0
...,...,...,...,...,...,...,...,...,...
1181,AKRPDCCP793ET,1731,304,B,q3,1,0,0,0
1180,AKRPDCCP793ET,1731,304,B,q2,1,0,0,0
1182,AKRPDCCP793ET,1731,306,A,q1,1,0,0,0
1183,AKRPDCCP793ET,1731,306,A,q2,1,0,0,0


In [107]:
df

,WorkerId,WorkTimeInSeconds,subj,speech,q_id,methodA,methodB,methodA_wins,methodB_wins,ours_wins,ga_wins,talkg_wins,instag_wins
0,A1WTHBAWHRVU4P,559,104,B,q1,ours,instag,True,False,1,0,0,0
1,A1WTHBAWHRVU4P,559,104,B,q2,ours,instag,True,False,1,0,0,0
2,A1WTHBAWHRVU4P,559,104,B,q3,ours,instag,True,False,1,0,0,0
3,A1WTHBAWHRVU4P,559,218,A,q1,ours,ga,True,False,1,0,0,0
4,A1WTHBAWHRVU4P,559,218,A,q2,ours,ga,True,False,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,A2KWD81ZH7T234,977,306,A,q2,ours,talkg,True,False,1,0,0,0
3236,A2KWD81ZH7T234,977,306,A,q3,ours,talkg,True,False,1,0,0,0
3237,A2KWD81ZH7T234,977,306,B,q1,talkg,ours,False,True,1,0,0,0
3238,A2KWD81ZH7T234,977,306,B,q2,talkg,ours,False,True,1,0,0,0


In [ ]:
# count how many times methodA has value 'ga' and methodB has value 'ga'
num_ga = df[(df['methodA'] == 'ga') | (df['methodB'] == 'ga')].shape[0]
num_ga

# 9 subj x 2 speech x 20 responses = 360 (per question)
# 9 subj x 2 speech x 20 responses x 3 questions = 1080 (total answers)

1080